In [ ]:
import pandas as pd
import os

In [ ]:
cwd= os.getcwd()
print(cwd)
BASE_DIR= os.path.join(cwd,"..", "..")

data= os.path.join(BASE_DIR, "data", "anemia-data.xlsx")

In [ ]:
#df= pd.read_csv(data)
df= pd.read_excel(data)
print(df)
print(df.shape)

In [ ]:
"""
def clean_df(df):
    a= df.drop(index= range(3))
    b= a.reset_index(drop= True)
    b.columns= b.iloc[0]
    c= b.drop(index= 0)
    d= c.drop(columns= "STT")
    e = d.iloc[:, :-4] # bỏ cột tên NaN
    return e

"""

In [ ]:
# cleaned_df= clean_df(df)
# print(cleaned_df.shape)

In [ ]:
# cleaned_df.info()
df.info()

In [ ]:
"""
text_cols = [1, 2, 23]

for i in range(len(cleaned_df.columns)):
    if i not in text_cols:
        col_name = cleaned_df.columns[i]
        # Gán lại bằng TÊN cột, không dùng iloc
        cleaned_df[col_name] = pd.to_numeric(cleaned_df[col_name], errors='coerce')
    else:
        col_name = cleaned_df.columns[i]
        cleaned_df[col_name] = cleaned_df[col_name].astype("string")



print(cleaned_df.info())
"""


In [ ]:
# cleaned_df.columns

### Univariable Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# thay cleaned_df khi có data bệnh viện
"""
for i in range(len(cleaned_df.columns)):
    if i not in text_cols:
        col_name = cleaned_df.columns[i]
        
        plt.figure()  # tạo figure mới
        sns.histplot(cleaned_df[col_name], bins=10)
        plt.show()

"""


In [ ]:

X_columns= df.columns[:-1]
for i in range(len(X_columns)):
    col_name= df.columns[i]
    plt.figure()
    sns.boxplot(data= df, x=col_name, hue= "Decision_Class")
    plt.legend(
    title="Diagnosis",
    bbox_to_anchor=(1.05, 1),  # đẩy sang phải
    loc='upper left'
)
    plt.show()

In [ ]:
df["Decision_Class_Label"] = df["Decision_Class"].map({0: "healthy", 1: "anemia"})
sns.countplot(data=df, x="Decision_Class_Label")
plt.xlabel("Diagnosis")
plt.ylabel("Samples")

In [ ]:
sns.countplot(data= df, x= "Gender", hue= "Decision_Class")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = df.drop("Diagnosis", axis=1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=df["Diagnosis"])

In [ ]:
from sklearn.manifold import TSNE

X = df.drop("Diagnosis", axis=1)
X_embedded = TSNE(n_components=2).fit_transform(X)

plt.figure()
sns.scatterplot(x=X_embedded[:,0], y=X_embedded[:,1], hue=df["Diagnosis"])
plt.legend(
    title="Diagnosis",
    bbox_to_anchor=(1.05, 1),  # đẩy sang phải
    loc='upper left'
)
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

score = silhouette_score(X_embedded, df["Diagnosis"])
print(score)

In [ ]:
plt.figure()

ax = sns.kdeplot(
    x=X_embedded[:,0],
    y=X_embedded[:,1],
    hue=df["Diagnosis"]
)

sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3)
clusters = kmeans.fit_predict(X_scaled)

sns.scatterplot(x=X_embedded[:,0], y=X_embedded[:,1], hue=clusters)

In [ ]:
df.groupby("Diagnosis")["LYMp"].describe()

In [ ]:
#sns.histplot(cleaned_df["Chẩn đoán (chỉ liên quan hồng cầu nhỏ: IDA, ACD, Thalassemia, CRNN)"])
sns.histplot(df["Diagnosis"])
plt.xticks(rotation=90)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

def combine_images(img_paths):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    for ax, img_path in zip(axes, img_paths):
        img = plt.imread(img_path)
        ax.imshow(img)
        ax.axis("off")  # tắt trục cho đẹp

    return fig

pr_paths= ["pr_curve_knn.png", "pr_curve_nb.png", "pr_curve_svm.png"]
roc_paths= ["roc_auc_knn.png", "roc_auc_nb.png", "roc_auc_svm.png"]



In [ ]:
combined_pr_fig = combine_images(pr_paths)


In [ ]:
combined_roc_fig = combine_images(roc_paths)

In [ ]:
from sklearn.model_selection import train_test_split

df = pd.get_dummies(df, columns=["Gender"], drop_first=True)
    
df = df.apply(pd.to_numeric, errors='coerce')

X = df.drop(columns=["Decision_Class"])
y = df["Decision_Class"]

X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y 
)

In [ ]:
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

def plot_multi_roc(models_dict, y_true, X_test):
    """
    models_dict: {
        "RF": path_model,
        "XGB": path_model,
        ...
    }
    """

    fig, ax = plt.subplots()

    for name, path in models_dict.items():
        model = joblib.load(path)
        y_prob = model.predict_proba(X_test)[:, 1]

        fpr, tpr, _ = roc_curve(y_true, y_prob)
        roc_auc = auc(fpr, tpr)

        ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.3f})")

    # đường baseline
    ax.plot([0, 1], [0, 1], linestyle="--")

    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.set_title("ROC Curve Comparison")
    ax.legend()

    return fig

def plot_multi_pr(models_dict, y_true, X_test):

    fig, ax = plt.subplots()

    for name, path in models_dict.items():
        model = joblib.load(path)
        y_prob = model.predict_proba(X_test)[:, 1]

        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        ap = average_precision_score(y_true, y_prob)

        ax.plot(recall, precision, label=f"{name} (AP={ap:.3f})")

    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("PR Curve Comparison")
    ax.legend()

    return fig

In [ ]:
models = {
    "KNN": os.path.join(BASE_DIR, "artifacts", "KNN_best_model.pkl"),
    "SVM": os.path.join(BASE_DIR, "artifacts", "SVM_best_model.pkl"),
    "NaiveBayes": os.path.join(BASE_DIR, "artifacts", "NaiveBayes_best_model.pkl")
}

fig_roc = plot_multi_roc(models, y_test, X_test)
fig_pr = plot_multi_pr(models, y_test, X_test)

In [ ]:
fig_pr = plot_multi_pr(models, y_test, X_test)
display(fig_pr)

In [ ]:
fig_roc = plot_multi_roc(models, y_test, X_test)
display(fig_roc)